<a href="https://colab.research.google.com/github/asoto59g/GEE_SM_Model/blob/main/GEE_SM_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 README - Metodología de Estimación de Humedad del Suelo (SM)

## 📌 Descripción general

Este script implementa una metodología multifuente para estimar la **humedad del suelo (Soil Moisture, SM)** utilizando datos satelitales y variables topográficas en un periodo multianual.

El modelo combina información de:

- Radar (Sentinel-1)
- Óptico (Sentinel-2)
- Topografía (SRTM)

El resultado es un índice normalizado de humedad del suelo en el rango **[0 – 1]**, donde:

- **0 → seco**
- **1 → alta humedad**

---

## 🧠 Enfoque metodológico

La estimación de humedad se basa en la integración de variables físicas relacionadas con la retención de agua en el suelo:

| Variable | Fuente | Relación con humedad |
|----------|--------|---------------------|
| SAR (VV, VH) | Sentinel-1 | Sensible a humedad superficial |
| NDMI | Sentinel-2 | Contenido de agua en vegetación |
| NDVI | Sentinel-2 | Cobertura vegetal |
| Pendiente | SRTM | Influye en escorrentía |

---

## ⚙️ Flujo del proceso

### 1. 📍 Datos de entrada

- Puntos de muestreo (FeatureCollection)
- Periodo de análisis: **2020 – 2026**
- Región: buffer de 5 km alrededor de los puntos

---

### 2. ⛰️ Topografía

- Fuente: SRTM (`USGS/SRTMGL1_003`)
- Derivación:
  - Pendiente (`slope`)
- Normalización:
  slope → [0 – 1]

---

### 3. 📡 Procesamiento SAR (Sentinel-1)

- Polarizaciones: `VV`, `VH`
- Filtros:
  - Modo IW
  - Órbita descendente
- Aplicación de filtro speckle:
  focal_mean (30 m)

- Cálculo:
  SAR = (VV + VH) / 2

- Normalización:
  SAR → [0 – 1] usando unitScale(-20, 0)

---

### 4. 🌿 Procesamiento óptico (Sentinel-2)

- Filtro de nubes: < 20%
- Composición: mediana temporal

#### Índices calculados:

- **NDVI (vegetación):**
  NDVI = (B8 - B4) / (B8 + B4)

- **NDMI (humedad):**
  NDMI = (B8 - B11) / (B8 + B11)

- Ambos normalizados a:
  [0 – 1]

---

### 5. 🧮 Modelo de humedad del suelo

SM = (SAR * 0.5)
   + (NDMI * 0.3)
   + ((1 - NDVI) * 0.1)
   + ((1 - SLOPE) * 0.1)

---

### 6. 📅 Análisis multianual

- Se ejecuta el modelo para cada año (2020–2026)
- Ventana temporal fija:
  15 marzo – 8 abril

---

### 7. 🗺️ Visualización

- Paleta:
  - Marrón → seco
  - Amarillo → intermedio
  - Azul → húmedo

---

### 8. 📈 Serie temporal

Se genera un gráfico con la evolución temporal de la humedad.

---

### 9. 📍 Muestreo espacial

- Extracción de valores en puntos
- Atributos:
  - Año
  - Latitud
  - Longitud

---

### 10. 💾 Exportaciones

#### 📊 CSV
Incluye variables y coordenadas.

#### 🗺️ Mapas
- Resolución: 10 m
- Carpeta: `GEE_Humedad`

---

## 🚀 Uso

1. Cargar script en GEE
2. Verificar assets
3. Ejecutar
4. Exportar resultados

---

## ⚠️ Consideraciones

- Modelo semi-empírico
- Sensible a condiciones atmosféricas

---

## 🔧 Mejoras futuras

- Calibración con campo
- Machine Learning
- Integración climática


# 🌎 Modelado multianual de humedad del suelo con GEE
---
Notebook de Google Colab adaptado del script en JavaScript (Code Editor)

- Autenticación y uso del proyecto **ee-oasotob**
- Procesamiento Sentinel‑1 + Sentinel‑2 + SRTM
- Visualización y exportaciones a Google Drive

In [ ]:
!pip install geemap --quiet
import ee, geemap

# Autenticación — se abrirá un enlace en el navegador
ee.Authenticate()
# Inicializar Earth Engine con el proyecto
ee.Initialize(project='ee-oasotob')
print('✅ Earth Engine inicializado con el proyecto ee-oasotob')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 38.4 MB/s eta 0:00:00
✅ Earth Engine inicializado con el proyecto ee-oasotob


In [ ]:
puntos = ee.FeatureCollection('projects/ee-oasotob/assets/ptsmuestreo')
years = list(range(2020, 2027))
region = puntos.geometry().buffer(5000)
print('Puntos cargados:', puntos.size().getInfo())

Puntos cargados: 53


In [ ]:
dem = ee.Image('USGS/SRTMGL1_003')
slope = ee.Terrain.slope(dem).rename('slope')

In [ ]:
def speckle_filter(img):
    return img.focal_mean(30, 'circle', 'meters')

In [ ]:
def modelo(year):
    year = ee.Number(year)
    start = ee.Date.fromYMD(year, 3, 15)
    end   = ee.Date.fromYMD(year, 4, 8)

    s1_raw = (ee.ImageCollection('COPERNICUS/S1_GRD')
              .filterDate(start, end)
              .filterBounds(region)
              .filter(ee.Filter.eq('instrumentMode', 'IW'))
              .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
              .select(['VV','VH']).limit(20).map(speckle_filter))
    s1 = s1_raw.mean().rename(['VV','VH'])
    sar = s1.expression('(VV + VH)/2', {'VV':s1.select('VV'),'VH':s1.select('VH')}).rename('SAR')
    sar_norm = sar.unitScale(-20,0).clamp(0,1)

    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterDate(start, end)
           .filterBounds(region)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',20))
           .median())

    ndvi = s2.normalizedDifference(['B8','B4']).rename('NDVI').clamp(0,1)
    ndmi = s2.normalizedDifference(['B8','B11']).rename('NDMI').clamp(0,1)
    slope_norm = slope.unitScale(0,30).clamp(0,1)

    sm = sar_norm.expression(
        '(SAR*0.5)+(NDMI*0.3)+((1-NDVI)*0.1)+((1-SLOPE)*0.1)',
        {'SAR':sar_norm,'NDMI':ndmi,'NDVI':ndvi,'SLOPE':slope_norm}
    ).rename('SM_final')

    return sm.addBands([ndvi, ndmi, slope]).set({'year':year,'system:time_start':start.millis()})

In [ ]:
coleccion = ee.ImageCollection([modelo(y) for y in years])
print('Colección creada:', coleccion.size().getInfo())

Colección creada: 7


In [ ]:
Map = geemap.Map(center=[10.469,-85.323], zoom=8)
vis = {'min':0,'max':1,'palette':['brown','yellow','blue']}
for y in years:
    img = coleccion.filter(ee.Filter.eq('year',y)).first()
    Map.addLayer(img.select('SM_final'), vis, f'Humedad {y}')
Map.addLayer(puntos, {'color':'black'}, 'Puntos muestreo')
Map.add_colorbar_branca(colors=['brown','yellow','blue'],caption='Humedad (0-1)')
Map

Map(center=[10.469, -85.323], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDat…

In [ ]:
def extraer_muestras(img):
    year = img.get('year')
    sampled = img.sampleRegions(collection=puntos, scale=10, geometries=True)
    def set_coords(f):
        coords = f.geometry().coordinates()
        return f.set({'year':year,'lon':coords.get(0),'lat':coords.get(1)})
    return sampled.map(set_coords)

muestras = coleccion.map(extraer_muestras).flatten()
task_tbl = ee.batch.Export.table.toDrive(collection=muestras, description='SM_Experto_FINAL', fileFormat='CSV')
task_tbl.start()
print('✅ Exportación de tabla iniciada')

for y in years:
    img = coleccion.filter(ee.Filter.eq('year', y)).first().select('SM_final')
    task_img = ee.batch.Export.image.toDrive(image=img, description=f'SM_{y}', folder='GEE_Humedad',
        fileNamePrefix=f'SM_{y}', region=region, scale=10, maxPixels=1e13)
    task_img.start()
print('✅ Exportaciones de imágenes iniciadas (Drive/GEE_Humedad)')